<a href="https://colab.research.google.com/github/Pranayshukla0610/ML-projects-portfolio/blob/main/Apple_Financial_Analytics_Dashboard_Using_Bokeh_Library_in_Python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import yfinance as yf
import pandas as pd
import numpy as np

from math import pi
from bokeh.io import curdoc
from bokeh.layouts import row, column
from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import(
    ColumnDataSource,
    HoverTool,
    Div
)

In [ ]:
output_notebook()

In [ ]:
ticker = 'AAPL'

data = yf.download(
    ticker,
    period="1y"
)

data.reset_index(inplace=True)

/tmp/ipykernel_1535/2347753420.py:3: FutureWarning: YF.download() has changed argument auto_adjust default to True
  data = yf.download(
[*********************100%***********************]  1 of 1 completed


In [ ]:
data.head()

Price,Date,Close,High,Low,Open,Volume
Ticker,,AAPL,AAPL,AAPL,AAPL,AAPL
0,2025-05-07,195.398376,198.574536,192.411395,198.305703,68536700
1,2025-05-08,196.633011,199.181899,193.835192,196.862009,50478900
2,2025-05-09,197.668472,199.669744,196.682763,198.136434,36453900
3,2025-05-12,210.150482,210.629037,206.122746,210.329944,63775800
4,2025-05-13,212.284012,212.752587,208.365942,209.791596,51909300


In [ ]:
data['Returns'] = data['Close'].pct_change()

In [ ]:
data['MA_20'] = data['Close'].rolling(20).mean()
data['MA_50'] = data['Close'].rolling(50).mean()

In [ ]:
data['Volatility'] = data['Returns'].rolling(20).std()

In [ ]:
source = ColumnDataSource(data)

In [ ]:
latest_price = round(data['Close'].iloc[-1],2)
avg_return = round(data['Returns'].mean()*100,2)
volatility = round(data['Volatility'].mean()*100,2)

kpi1 = Div(text=f"""
<div style="
background-color:#111827;
padding:20px;
border-radius:12px;
text-align:center;
color:white;
width:250px;
">
<h2>Latest Price</h2>
<h1>${latest_price}</h1>
</div>
""")

kpi2 = Div(text=f"""
<div style="
background-color:#2563EB;
padding:20px;
border-radius:12px;
text-align:center;
color:white;
width:250px;
">
<h2>Average Return</h2>
<h1>{avg_return}%</h1>
</div>
""")

kpi3 = Div(text=f"""
<div style="
background-color:#DC2626;
padding:20px;
border-radius:12px;
text-align:center;
color:white;
width:250px;
">
<h2>Volatility</h2>
<h1>{volatility}%</h1>
</div>
""")

In [ ]:
from bokeh.io import output_notebook, show
# Remove MultiIndex columns
data.columns = data.columns.get_level_values(0)

# Boolean conditions
inc = data['Close'] > data['Open']
dec = data['Open'] > data['Close']

# Width
w = 12 * 60 * 60 * 1000

# Figure
p1 = figure(
    x_axis_type="datetime",
    width=1000,
    height=400,
    title="Candlestick Chart",
    background_fill_color="#1F2937",
    border_fill_color="#111827"
)

# High-Low line
p1.segment(
    data['Date'],
    data['High'],
    data['Date'],
    data['Low'],
    color="white"
)

# Green candles
p1.vbar(
    x=data.loc[inc, 'Date'],
    width=w,
    top=data.loc[inc, 'Close'],
    bottom=data.loc[inc, 'Open'],
    fill_color="#10B981",
    line_color="#10B981"
)

# Red candles
p1.vbar(
    x=data.loc[dec, 'Date'],
    width=w,
    top=data.loc[dec, 'Open'],
    bottom=data.loc[dec, 'Close'],
    fill_color="#EF4444",
    line_color="#EF4444"
)

# Hover tool
hover = HoverTool(
    tooltips=[
        ("Date", "@Date{%F}"),
        ("Open", "@Open"),
        ("Close", "@Close"),
        ("High", "@High"),
        ("Low", "@Low")
    ],
    formatters={
        '@Date': 'datetime'
    }
)

p1.add_tools(hover)



In [ ]:
p2 = figure(
    x_axis_type='datetime',
    width=1000,
    height=400,
    title="Moving Averages",
    background_fill_color="#1F2937",
    border_fill_color="#111827"
)

p2.line(
    data['Date'],
    data['Close'],
    line_width=2,
    color='white',
    legend_label='Close Price'
)

p2.line(
    data['Date'],
    data['MA_20'],
    line_width=3,
    color='blue',
    legend_label='MA 20'
)

p2.line(
    data['Date'],
    data['MA_50'],
    line_width=3,
    color='orange',
    legend_label='MA 50'
)

GlyphRenderer(id='p2521', ...)

In [ ]:
p3 = figure(
    x_axis_type='datetime',
    width=1000,
    height=300,
    title="Volatility Analysis",
    background_fill_color="#1F2937",
    border_fill_color="#111827"
)

p3.line(
    data['Date'],
    data['Volatility'],
    line_width=3,
    color='red'
)

GlyphRenderer(id='p2582', ...)

In [ ]:
investment = 10000

shares = investment / data['Close'].iloc[0]

portfolio_value = shares * data['Close'].iloc[-1]

profit = portfolio_value - investment

In [ ]:
portfolio_kpi = Div(text=f"""
<div style="
background-color:#059669;
padding:20px;
border-radius:12px;
text-align:center;
color:white;
width:350px;
">
<h2>Portfolio Value</h2>
<h1>${round(portfolio_value,2)}</h1>

<h2>Profit/Loss</h2>
<h1>${round(profit,2)}</h1>
</div>
""")

In [ ]:
dashboard = column(

    Div(text=f"""
    <h1 style="
    color:white;
    background-color:#111827;
    padding:20px;
    border-radius:10px;
    text-align:center;
    ">
    REAL-TIME FINANCIAL ANALYTICS DASHBOARD
    </h1>
    """),

    row(
        kpi1,
        kpi2,
        kpi3,
        portfolio_kpi
    ),

    p1,
    p2,
    p3
)

In [ ]:
def update():

    global data

    new_data = yf.download(
        ticker,
        period="1d",
        interval="1m"
    )

    latest_price = new_data['Close'].iloc[-1]

    source.stream({
        'Date':[pd.Timestamp.now()],
        'Close':[latest_price]
    }, rollover=100)



In [ ]:
from bokeh.layouts import column

dashboard = column(p1, p2, p3)

In [ ]:
from bokeh.io import curdoc
curdoc().clear()

In [ ]:
show(dashboard)

In [ ]:
from bokeh.io import output_file, save

output_file("dashboard.html")

save(dashboard)

show(dashboard)

In [ ]:
from google.colab import files

files.download("dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>